In [1]:
import pandas as pd
import numpy as np

In [8]:
# XLSX-Datei einlesen
df = pd.read_excel(r'C:\Users\miria\creditcard_psp\data\raw\PSP_Jan_Feb_2019.xlsx', index_col=0)

# Anzahl Zeilen
print(df.count())

# Anzahl Duplikate
duplicates = df.duplicated().sum()
print("Anzahl Duplikate: ", duplicates)

# Duplikate löschen
df = df.drop_duplicates()

# Anzahl Zeilen
print(df.count())

tmsp          50410
country       50410
amount        50410
success       50410
PSP           50410
3D_secured    50410
card          50410
dtype: int64
Anzahl Duplikate:  81
tmsp          50329
country       50329
amount        50329
success       50329
PSP           50329
3D_secured    50329
card          50329
dtype: int64


In [9]:
# Timestamp auf Minuten runden
df['tmsp_min'] = pd.to_datetime(df['tmsp'], errors='coerce').dt.floor('T')

# Gruppierungsschlüssel definieren
group_cols = ['tmsp_min', 'country', 'amount']

# Gesamt-Erfolgsflag pro Transaktion: max() auf 0/1 ergibt 1, wenn irgendwo ein 1 ist
success_df = (
    df
    .groupby(group_cols)['success']
    .max()
    .reset_index(name='transaction_success')
)

# Statische Features: pro Transaktion eine Zeile mit allen originalen Spalten (inkl. success, falls du sie brauchst)
static_df = df.drop_duplicates(subset=group_cols).copy()

# Zusammenführen
agg_df = static_df.merge(success_df, on=group_cols, how='left')

# Die ursprüngliche Spalte 'success' auf Transaktionsebene entfernen,
# weil sie jetzt redundant ist.
agg_df = agg_df.drop(columns=['success'])

# Ergebnis anschauen
agg_df.head()

,tmsp,country,amount,PSP,3D_secured,card,tmsp_min,transaction_success
0,2019-01-01 00:01:11,Germany,89,UK_Card,0,Visa,2019-01-01 00:01:00,1
1,2019-01-01 00:02:49,Germany,238,UK_Card,1,Diners,2019-01-01 00:02:00,0
2,2019-01-01 00:03:13,Germany,238,UK_Card,1,Diners,2019-01-01 00:03:00,1
3,2019-01-01 00:04:33,Austria,124,Simplecard,0,Diners,2019-01-01 00:04:00,0
4,2019-01-01 00:06:41,Switzerland,282,UK_Card,0,Master,2019-01-01 00:06:00,0
